In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

df = spark.read.format("parquet").load("abfss://bronze@dbproject.dfs.core.windows.net/products")
df.display()

product_id,product_name,category,brand,price,_rescued_data
P0001,Clearly Its,Beauty,Nike,1868.54,null
P0002,Production Clear,Beauty,Apple,587.13,null
P0003,Culture Coach,Home,Revlon,1599.24,null
P0004,Movement Part,Sports,LG,651.71,null
P0005,Fact Name,Clothing,Samsung,1861.78,null
P0006,Usually Stop,Toys,Adidas,936.36,null
P0007,Reveal Current,Sports,Adidas,1954.02,null
P0008,Force Language,Beauty,Puma,1251.26,null
P0009,Stage Leg,Clothing,Samsung,1247.15,null
P0010,Leader Then,Sports,Sony,975.53,null


In [0]:
df = df.drop('_rescued_data')
df.display()

product_id,product_name,category,brand,price
P0001,Clearly Its,Beauty,Nike,1868.54
P0002,Production Clear,Beauty,Apple,587.13
P0003,Culture Coach,Home,Revlon,1599.24
P0004,Movement Part,Sports,LG,651.71
P0005,Fact Name,Clothing,Samsung,1861.78
P0006,Usually Stop,Toys,Adidas,936.36
P0007,Reveal Current,Sports,Adidas,1954.02
P0008,Force Language,Beauty,Puma,1251.26
P0009,Stage Leg,Clothing,Samsung,1247.15
P0010,Leader Then,Sports,Sony,975.53


**Creating a view(Pyspark to SQL)**

In [0]:
df.createOrReplaceTempView("products")

**Creating a function to store in catalog**

In [0]:
%sql

CREATE OR REPLACE FUNCTION db_catalog.bronze.discount_func(p_price DOUBLE) 
RETURNS DOUBLE  
LANGUAGE SQL 
RETURN round(p_price * 0.7, 2)


In [0]:
df = df.withColumn("discounted_price",expr("db_catalog.bronze.discount_func(price)"))
df.display()

product_id,product_name,category,brand,price,discounted_price
P0001,Clearly Its,Beauty,Nike,1868.54,1307.98
P0002,Production Clear,Beauty,Apple,587.13,410.99
P0003,Culture Coach,Home,Revlon,1599.24,1119.47
P0004,Movement Part,Sports,LG,651.71,456.2
P0005,Fact Name,Clothing,Samsung,1861.78,1303.25
P0006,Usually Stop,Toys,Adidas,936.36,655.45
P0007,Reveal Current,Sports,Adidas,1954.02,1367.81
P0008,Force Language,Beauty,Puma,1251.26,875.88
P0009,Stage Leg,Clothing,Samsung,1247.15,873.01
P0010,Leader Then,Sports,Sony,975.53,682.87


We can query using SQL also

In [0]:
%sql
select product_id, price ,db_catalog.bronze.discount_func(price) as discounted_price
FROM products

product_id,price,discounted_price
P0001,1868.54,1307.98
P0002,587.13,410.99
P0003,1599.24,1119.47
P0004,651.71,456.2
P0005,1861.78,1303.25
P0006,936.36,655.45
P0007,1954.02,1367.81
P0008,1251.26,875.88
P0009,1247.15,873.01
P0010,975.53,682.87


In [0]:
%sql

CREATE OR REPLACE FUNCTION db_catalog.bronze.price_ctg(dis_p FLOAT)
RETURNS STRING
LANGUAGE PYTHON
AS
$$
    if dis_p < 500:
        return "Low"
    elif dis_p < 1000:
        return "Medium"
    else:
        return "High"
$$;


In [0]:
df = df.withColumn("price_category", expr("db_catalog.bronze.price_ctg(discounted_price)"))
df.display()

product_id,product_name,category,brand,price,discounted_price,price_category
P0001,Clearly Its,Beauty,Nike,1868.54,1307.98,High
P0002,Production Clear,Beauty,Apple,587.13,410.99,Low
P0003,Culture Coach,Home,Revlon,1599.24,1119.47,High
P0004,Movement Part,Sports,LG,651.71,456.2,Low
P0005,Fact Name,Clothing,Samsung,1861.78,1303.25,High
P0006,Usually Stop,Toys,Adidas,936.36,655.45,Medium
P0007,Reveal Current,Sports,Adidas,1954.02,1367.81,High
P0008,Force Language,Beauty,Puma,1251.26,875.88,Medium
P0009,Stage Leg,Clothing,Samsung,1247.15,873.01,Medium
P0010,Leader Then,Sports,Sony,975.53,682.87,Medium


In [0]:
df.write.format("delta").mode("overwrite").save("abfss://silver@dbproject.dfs.core.windows.net/products")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS db_catalog.silver.products_silver
USING DELTA
LOCATION 'abfss://silver@dbproject.dfs.core.windows.net/products'